# Parivahan RAG — Indexing & Evaluation Notebook
Run cells top-to-bottom to build index and test retrieval.

In [ ]:
import sys, os
from pathlib import Path
# Set working directory to project root (notebook lives in notebooks/)
ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv; load_dotenv()
print('CWD:', os.getcwd())

## 1. Data Pipeline — Download & Parse PDFs

In [ ]:
from src.data_pipeline import build_chunks
chunks = build_chunks()
print(f'Total chunks: {len(chunks)}')
print('Sample chunk:')
print(chunks[0])

## 2. Embeddings — Build FAISS Index

In [ ]:
from src.embeddings import load_model, embed_chunks, build_index, save_index
model = load_model()
embs  = embed_chunks(model, chunks)
index = build_index(embs)
save_index(index, chunks)
print(f'Index: {index.ntotal} vectors, dim={index.d}')

## 3. Retrieval Smoke Test

In [ ]:
from src.embeddings import search, load_index
index, meta = load_index()

TEST_QUERIES = [
    'How to apply for driving licence in India?',
    'Documents required for RC registration',
    'Vehicle ownership transfer procedure',
    'What is learner licence test fee?',
]

for q in TEST_QUERIES:
    results = search(q, model, index, meta, top_k=3)
    print(f'\nQuery: {q}')
    for r in results:
        print(f'  [{r["source"]} p{r["page"]}] score={r["score"]:.3f} | {r["text"][:100]}…')

## 4. End-to-End RAG Test

In [ ]:
from src.rag_pipeline import answer_query, format_citations
query = 'How do I apply for Driving License and RC in India?'
retrieved = search(query, model, index, meta, top_k=5)
answer, srcs = answer_query(query, retrieved, hindi=False)
print('ANSWER:\n', answer)
print('\nCITATIONS:\n', format_citations(srcs))

## 5. Hindi Mode Test

In [ ]:
answer_hi, srcs_hi = answer_query(query, retrieved, hindi=True)
print('HINDI ANSWER:\n', answer_hi)

## 6. Ablation — Effect of Top-K on Answer Quality

In [ ]:
q = 'What documents do I need for RC transfer?'
for k in [1, 3, 5, 8]:
    hits = search(q, model, index, meta, top_k=k)
    ans, _ = answer_query(q, hits)
    print(f'\n--- top_k={k} ---')
    print(ans[:300], '…')